In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp -r /content/drive/MyDrive/split_dataset_balanced /content/

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

cuda
Tesla T4


In [ ]:
train_dir = '/content/split_dataset_balanced/train'
val_dir = '/content/split_dataset_balanced/val'

print(os.path.exists(train_dir), train_dir)
print(os.path.exists(val_dir), val_dir)

True /content/split_dataset_balanced/train
True /content/split_dataset_balanced/val


In [ ]:
train_ds = datasets.ImageFolder(train_dir, transform=train_tfms)
val_ds = datasets.ImageFolder(val_dir, transform=val_tfms)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

class_names = train_ds.classes

print(class_names)
print("Train samples:", len(train_ds))
print("Val samples:", len(val_ds))

['destroyed', 'major-damage', 'minor-damage', 'no-damage']
Train samples: 2064
Val samples: 520


In [ ]:
img_size = 224
batch_size = 32

train_tfms = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
])

val_tfms = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
])

In [ ]:
class HybridCNNResNet(nn.Module):
    def __init__(self, num_classes=4):
        super(HybridCNNResNet, self).__init__()

        self.cnn_branch = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((1, 1))
        )

        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.resnet_branch = nn.Sequential(*list(resnet.children())[:-1])

        self.classifier = nn.Sequential(
            nn.Linear(128 + 2048, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        cnn_feat = self.cnn_branch(x)
        cnn_feat = torch.flatten(cnn_feat, 1)

        resnet_feat = self.resnet_branch(x)
        resnet_feat = torch.flatten(resnet_feat, 1)

        fused = torch.cat((cnn_feat, resnet_feat), dim=1)
        out = self.classifier(fused)
        return out

In [ ]:
model = HybridCNNResNet(num_classes=4).to(device)
print(model)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 183MB/s]


HybridCNNResNet(
  (cnn_branch): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): AdaptiveAvgPool2d(output_size=(1, 1))
  )
  (resnet_branch): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): Bottleneck(
        (conv1): 

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')

    return epoch_loss, epoch_acc, epoch_f1


def validate_one_epoch(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')

    return epoch_loss, epoch_acc, epoch_f1, all_labels, all_preds

In [ ]:
num_epochs = 5
best_f1 = 0.0
save_path = '/content/best_hybrid_cnn_resnet_balanced.pth'

for epoch in range(num_epochs):
    train_loss, train_acc, train_f1 = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc, val_f1, y_true, y_pred = validate_one_epoch(model, val_loader, criterion)

    scheduler.step(val_f1)

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), save_path)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Train F1: {train_f1:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f} | Val   F1: {val_f1:.4f}")
    print("-" * 60)

Epoch 1/5
Train Loss: 1.1284 | Train Acc: 0.5053 | Train F1: 0.5156
Val   Loss: 0.9119 | Val   Acc: 0.6442 | Val   F1: 0.6456
------------------------------------------------------------
Epoch 2/5
Train Loss: 0.8125 | Train Acc: 0.6788 | Train F1: 0.6785
Val   Loss: 0.8262 | Val   Acc: 0.6750 | Val   F1: 0.6751
------------------------------------------------------------
Epoch 3/5
Train Loss: 0.6901 | Train Acc: 0.7238 | Train F1: 0.7246
Val   Loss: 0.7715 | Val   Acc: 0.6981 | Val   F1: 0.6919
------------------------------------------------------------
Epoch 4/5
Train Loss: 0.6089 | Train Acc: 0.7582 | Train F1: 0.7580
Val   Loss: 0.7360 | Val   Acc: 0.7212 | Val   F1: 0.7193
------------------------------------------------------------
Epoch 5/5
Train Loss: 0.5136 | Train Acc: 0.7975 | Train F1: 0.7971
Val   Loss: 0.8141 | Val   Acc: 0.6846 | Val   F1: 0.6749
------------------------------------------------------------


In [ ]:
model.load_state_dict(torch.load(save_path))

_, val_acc, val_f1, y_true, y_pred = validate_one_epoch(model, val_loader, criterion)

precision = precision_score(y_true, y_pred, average='macro')
recall = recall_score(y_true, y_pred, average='macro')
f1 = f1_score(y_true, y_pred, average='macro')

print("Best Validation Accuracy:", val_acc)
print("Best Validation Precision:", precision)
print("Best Validation Recall:", recall)
print("Best Validation Macro F1:", f1)

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_true, y_pred))

Best Validation Accuracy: 0.7211538461538461
Best Validation Precision: 0.7232564602526838
Best Validation Recall: 0.7211538461538463
Best Validation Macro F1: 0.7193292582210338

Classification Report:

              precision    recall  f1-score   support

   destroyed       0.93      0.89      0.91       130
major-damage       0.66      0.54      0.59       130
minor-damage       0.62      0.65      0.64       130
   no-damage       0.68      0.80      0.74       130

    accuracy                           0.72       520
   macro avg       0.72      0.72      0.72       520
weighted avg       0.72      0.72      0.72       520


Confusion Matrix:

[[116   5   6   3]
 [  3  70  33  24]
 [  2  22  85  21]
 [  4   9  13 104]]


In [ ]:
print("\nFINAL METRICS")
print(f"Accuracy  : {val_acc:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")


FINAL METRICS
Accuracy  : 0.7212
Precision : 0.7233
Recall    : 0.7212
F1 Score  : 0.7193
